In [1]:
"""
Importation necessaire:
WhisperProcessor: - Pour convertir l'audio en en spectrogrammes log-Mel.
                  - Pour transcire les tokens generes par le decodeur en texte.
WhisperForConditionalGeneration: - Pour telecharger le model whisper et l'instancier
load_dataset: - Pour charger un dataset, elle retourne un dict de deux cles, train contient les colonnes et num_arrows le nombre de lignes.
Audio: - Pour transformer un fichier audio en un tableau numpy
"""
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio

import evaluate

import os
import torch

/info/etu/m1/s2100786/miniconda3/envs/asr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Remarque:
- Les datasets commonvoice utilise dans les scripts fourni par huggingface ne sont plus supporter par load_dataset  
  SOLUTION: On charge les donnees sur le Disk et on les recupere avec load_dataset et on fait un petit traitement pour utiliser
  le script sans le modifier.  
- Pour l'inference en anglais, Le script utilise un dataset qui ne pose pas ce probleme.  


In [2]:
student_id = !echo "$USER"

DATASETS_PATH = f"/info/raid-etu/m1/{student_id[0]}/projet-m1-asr/datasets/"
DATASETS_PATH

'/info/raid-etu/m1/s2100786/projet-m1-asr/datasets/'

In [3]:
def load_and_process_data (DATASETS_PATH, FOLDER, FILE_NAME):
    # Charger le dataset
    dataset = load_dataset('csv', data_files=os.path.join("..", "datasets", FOLDER, f"{FILE_NAME}.tsv"), sep="\t")
    PATH_TO_AUDIO = DATASETS_PATH + FOLDER + "clips/"
    # Creer la colonne audio qui contient le chemin complet + le nom de fichier qui reside dans la colonne path
    dataset["train"] = dataset["train"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})
    """
    cast_columns: change le type d'une colonne
    Audio: permet de transformer un fichier audio en un tableau numpy avece un echantillonage de 16000 khz
    donc la colonne audio contiendra le tableau numpy qui sera apres transformer en spectogramme mel-log
    """
    dataset["train"] = dataset["train"].cast_column("audio", Audio(sampling_rate=16_000))
    return dataset

### Inférence pour toutes les données du dossier donné

In [18]:
def whisper_inference (DATASETS_PATH, FOLDER, FILE_NAME, TASK="transcribe", language="multilingue", nb=10):

    #device = "cuda" if torch.cuda.is_available() else "cpu"
    #print(f"Utilisation de: {device}")

    
    # get the WordErrorRate module and the CharacterErrorRate module
    wer_metric = evaluate.load("wer", module_type="metric")
    cer_metric = evaluate.load("cer", module_type="metric")

    
    # load model and processor
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    #model = model.to(device)
   
    if language == "multilingue":
        model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(task=TASK)
    else:
        model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

    # Pour stocker les resultats
    results = []

    # Pour transcire tous les audio
    iterator_ds = iter(ds["train"])

    for la_data in iterator_ds:
        # recuperer la colonne audio qui contient le tableau numpy qui represente l'audio
        input_speech = la_data["audio"]

        # transformer le tableau numpy en un spectogramme log-mel
        input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features
        # generer les tokens
        predicted_ids = model.generate(input_features)
        # decoder les tokens (transcire)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        # stocker les resultats
        results.append({
            "path": la_data["path"], # chemin complet
            "sentence": la_data["sentence"],  # transcription reelle (etiquette)
            "predicted": transcription,  # transcription Whisper    
            "wer": wer_metric.compute(references=[la_data["sentence"]], predictions=[transcription]),   # calcule le      Word Error Rate pour chaque data
            "cer": cer_metric.compute(references=[la_data["sentence"]], predictions=[transcription])    # calcule le Character Error Rate pour chaque data
        })

        wer_metric.add(references=la_data["sentence"], predictions=transcription)    # calcule le      Word Error Rate pour chaque data
        cer_metric.add(references=la_data["sentence"], predictions=transcription)    # calcule le Character Error Rate pour chaque data

    return results, wer_metric, cer_metric

### Inférence pour un nombre "nb" de données

In [5]:
def whisper_inference_pour_quelques_lignes (DATASETS_PATH, FOLDER, FILE_NAME, TASK="transcribe", LANGUAGE="multilingue", nb=10):

    #device = "cuda" if torch.cuda.is_available() else "cpu"
    #print(f"Utilisation de: {device}")

    
    # get the WordErrorRate module and the CharacterErrorRate module
    wer_metric = evaluate.load("wer", module_type="metric")
    cer_metric = evaluate.load("cer", module_type="metric")

    
    # load model and processor
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    #model = model.to(device)
    
    if LANGUAGE == "multilingue":
        model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(task=TASK)
    else:
        model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    
    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

    data = ds["train"]
    # Pour stocker les resultats
    results = []

    for ii in range(nb):
        # recuperer la colonne audio qui contient le tableau numpy qui represente l'audio
        input_speech = data[ii]["audio"]

        # transformer le tableau numpy en un spectogramme log-mel
        input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features
        # generer les tokens
        predicted_ids = model.generate(input_features)
        # decoder les tokens (transcire)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # stocker les resultats
        results.append({
            "path": data[ii]["path"], # chemin complet
            "sentence": data[ii]["sentence"],  # transcription reelle (etiquette)
            "predicted": transcription,  # transcription Whisper    
            "wer": wer_metric.compute(references=[data[ii]["sentence"]], predictions=[transcription]),   # calcule le      Word Error Rate pour chaque data
            "cer": cer_metric.compute(references=[data[ii]["sentence"]], predictions=[transcription])    # calcule le Character Error Rate pour chaque data
        })

        wer_metric.add(references=data[ii]["sentence"], predictions=transcription)    # ajoute les références et prédictions pour chaque audio
        cer_metric.add(references=data[ii]["sentence"], predictions=transcription)    # ajoute les références et prédictions pour chaque audio
        

    return results, wer_metric, cer_metric

### calcul des métriques WER CER

In [6]:
# WER et CER calculées pendant l'inférence
def afficher_WER_CER_pour_chaque_donnees(results, display_path=True, display_predicted=True, display_reference=True, display_WER=True, display_CER=True):
    # affichage
    for r in results:
        if display_path:
            print(f"Path: {r['path']}")
        if display_predicted:
            print(f"Whisper: {r['predicted']}")
        if display_reference:
            print(f"Référence: {r['sentence']}")
        if display_WER:
            print(f"WER: {r['wer']}")
        if display_CER:
            print(f"CER: {r['cer']}")
        print("") # newline

In [7]:
# moyenne des WER et CER qui ont été calculées pendant l'inférence
def afficher_moyenne_globale_WER_CER(results):
    wer_sum = cer_sum = 0
    n = len(results)
    
    for r in results:
        wer_sum += r['wer']
        cer_sum += r['cer']

    # affichage
    print(f"Moyenne WordErrorRate: {wer_sum/n}")
    print(f"Moyenne CharErrorRate: {cer_sum/n}")

In [8]:
# "post inférence" parce que les métriques sont calculées après l'inférence, et non pendant
def afficher_calcul_WER_CER_post_inference(wer_metric, cer_metric):
    print(f"Calcul de WER à la fin: {wer_metric.compute()}")
    print(f"Calcul de CER à la fin: {cer_metric.compute()}")

### Paramètres pour Inference de Zazaki/test.tsv en Francais

In [9]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"

### Paramètres pour Inference de Zazaki/test.tsv en Anglais

In [ ]:
LANGUAGE = "en"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"

### Paramètres pour Inference de Zazaki/test.tsv en Arabe

In [8]:
LANGUAGE = "ar"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "validated"

### Paramètres pour Inference de Turk/ss-corpus-tr.tsv en Turk

In [9]:
LANGUAGE = "turkish"
TASK = "transcribe"
FOLDER = "scripted_turkish/"
FILE_NAME = "dev"

### Paramètres pour Inference de Zazaki/test.tsv en Turc

In [8]:
LANGUAGE = "tr"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"

### Lancer l'Inférence

#### pour toutes les données

In [17]:
print("transcrire tous les audio : ")
results, wer_metric, cer_metric = whisper_inference(DATASETS_PATH, FOLDER, FILE_NAME, TASK, LANGUAGE, nb)

transcrire tous les audio : 


KeyboardInterrupt: 

#### pour x données

In [10]:
nb=3
print(f"transcrire {nb} audios : ")
results, wer_metric, cer_metric = whisper_inference_pour_quelques_lignes(DATASETS_PATH, FOLDER, FILE_NAME, TASK, LANGUAGE, nb)
print("fini")

transcrire 3 audios : 


Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


fini


## Et en multilingue

### Paramètres pour Inference de Zazaki/test.tsv par Whisper Multilangue

In [52]:
# il faut supprimer le paramètre LANGUAGE de l'appel d'une fonction
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"

### Lancer l'Inférence multilingue

#### pour toutes les données

In [ ]:
print("transcrire tous les audio : ")
results, wer_metric, cer_metric = whisper_inference (DATASETS_PATH, TASK, FOLDER, FILE_NAME)

#### pour x données

In [54]:
nb=3
print(f"transcrire {nb} audios...")
results, wer_metric, cer_metric = whisper_inference_pour_quelques_lignes(DATASETS_PATH, FOLDER, FILE_NAME, TASK, nb)
print("fini")

transcrire 3 audios : 
fini


### Résultats des métriques

In [ ]:
afficher_moyenne_globale_WER_CER(results)

In [ ]:
afficher_calcul_WER_CER_post_inference(wer_metric, cer_metric)

In [13]:
afficher_WER_CER_pour_chaque_donnees(results, 
                                     display_path=False, 
                                     display_predicted=True, 
                                     display_reference=True, 
                                     display_WER=True, 
                                     display_CER=True)

Whisper:  Ne derler bilirsiniz.
Référence: Ne derler bilirsiniz.
WER: 0.0
CER: 0.0

Whisper:  Macide bir şey anlamadan şaşkın şaşkın dinliyor.
Référence: Macide bir şey anlamadan şaşkın şaşkın dinliyordu.
WER: 0.1
CER: 0.028169014084507043

Whisper:  dışarı çıkalım
Référence: Dışarı çıkalım.
WER: 0.3333333333333333
CER: 0.06153846153846154



### Résultats d'exécutions

#### Résultats individuels pour Inference de Zazaki/test.tsv en Turc

In [9]:
afficher_resultats_individuels(results)

Whisper:  Zavka fil yaya.
label: Ez zaf qefilîyaya
wer: 1.0
cer: 0.5294117647058824
Whisper:  Doha Bandsı
label: Doxe biance!
wer: 1.0
cer: 0.5833333333333334
Whisper:  Pöveviya Rojane Uyshan
label: Bi hêvîya rojanê weşan
wer: 1.0
cer: 0.5454545454545454
Whisper:  Mövverinan unemeyan rasut krot.
label: Mi verênan û neweyan ra sûd girewt
wer: 1.0
cer: 0.4117647058823529
Whisper:  Götü havalı ama ne olur.
label: Azad hevalê min o.
wer: 1.25
cer: 0.7222222222222222
Whisper:  Merhaba.
label: Mi va
wer: 1.0
cer: 1.2
Whisper:  Tükür varna az zanana
label: Çi ke vana, ez zanena
wer: 1.0
cer: 0.42857142857142855
Whisper:  Mahalli Mahalli Kırmanca
label: Malima mi kirmanc a
wer: 1.0
cer: 0.631578947368421
Whisper:  Be roda isk'a ümiyabi
label: Pêrodayîş qewimîyabî
wer: 2.0
cer: 0.65
Whisper:  O hakkımı verir diye ayki müverir aşığı.
label: Awa ke mı viri de biye, a ki mı viri ra şiye.
wer: 1.0
cer: 0.4888888888888889
Whisper:  Ne like dengizde az ne kerdom
label: Ni layıki dengız de azne kerdo.

#### Résultats individuels pour Inference de Zazaki/test.tsv par Whisper Multilangue

In [24]:
afficher_WER_CER_pour_chaque_donnees(results, 
                                     display_path=False, 
                                     display_predicted=True, 
                                     display_reference=True, 
                                     display_WER=True, 
                                     display_CER=True)

Whisper:  The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The The Th